# Informe ejecutivo — Pronóstico de cancelación de clientes

- **Interconnect:** telecomunicaciones
- **Fecha:** 19 de abril de 2026
- **Estado del proyecto:** Fase 2 concluida. Modelo entrenado y tuneado, lista de retención operativa.

## 1. Resumen ejecutivo

Interconnect ha perdido alrededor de **2.7 millones de dólares** por cancelaciones de clientes, equivalentes al **15 % del valor total** de su cartera. Dos terceras partes de esa fuga provienen del segmento de contrato mensual, donde la tasa de cancelación pasó del 28 % en 2014 al 55 % en 2019.

Se entrenaron 8 modelos (Dummy baseline + 7 algoritmos), XGBoost además con tuning de hiper patametros, y un ensemble por promedio de los 3 mejores por validación cruzada. El ganador es el **ensemble por promedio de los 3 mejores** (XGBoost tuneado + CatBoost + Random Forest), con **AUC 0.847 en prueba** y **0.859 en validación cruzada 5-fold**: ordena correctamente un 85 % de los pares de clientes por riesgo real.  El mejor modelo individual es **XGBoost tuneado** (0.848 en prueba). Como entregable, se generó la **lista de retención de marketing** con los 5 174 clientes activos segmentados en tres niveles (Alto, Medio, Bajo) con la acción recomendada por segmento.

La **recomendación** sigue siendo migrar a clientes mensuales hacia contratos anuales. La lista de retención permite **priorizar a quien contactar primero**.


## 2. Problema de negocio

Interconnect pierde clientes cada mes y con ellos pierde ingresos. Hoy la compañía reacciona cuando el cliente ya canceló, cuando ya es tarde para evitarlo. La pregunta es: ¿qué clientes actuales están por cancelar y en qué orden conviene contactarlos?

El valor del trabajo no está en predecir sí o no sobre cada cliente, sino en **priorizar la lista de contacto** del equipo de marketing. La capacidad del equipo es limitada y cada promoción tiene un costo. Un buen modelo de Machine lerning permite gastar el presupuesto en los clientes que realmente lo requieren y evita ofrecer descuentos a quienes no iban a cancelar.

## 3. Datos utilizados

La información se encuentra en cuatro archivos de Interconnect, unidos por el identificador de cada cliente.

| Fuente    | Que contiene                                                        | Clientes cubiertos |
|-----------|---------------------------------------------------------------------|-------------------:|
| Contract  | Tipo de contrato, fechas de alta y baja, cargos mensuales y totales | 7,043              |
| Personal  | Datos demográficos: género, edad, familia                           | 7,043              |
| Internet  | Servicio de internet y complementos (seguridad, respaldo, soporte)  | 5,517              |
| Phone     | Líneas telefonicas                                                  | 6,361              |

- **Volumen total:** 7,043 clientes unicos.
- **Período:** altas desde octubre de 2013 y **fecha de corte el 1 de febrero de 2020**.
- **Cobertura en internet y telefono:** no es información faltante. Son clientes que simplemente no contrataron esos servicios.

**Valore nulos :** Se revisaron las nueve columnas con valores vacíos. El resultado confirma que **no hay datos realmente perdidos**: los 1,526 nulos en columnas de internet son clientes sin servicio de internet; los 682 en `MultipleLines` son clientes sin el servicio; los 11 en `TotalCharges` son clientes nuevos, recien dados de alta sin historial de cobros.

## 4. Enfoque metodológico

El trabajo se hizo en seis pasos.

1. **EDA.** Una Exploración de datos inical para compronder la esctuctura de los datos.
2. **Limpieza.** Se realizo una exploracion de los datos faltantes, duplicados y homogeanizacion de los nombres en las columnas 
3. **Consolidación.** Las cuatro fuentes se unieron en una sola tabla, un renglón por cliente.
4. **Variables de negocio.** A los datos se sumaron tres columnas con sentido comercial:
   - Cancelación (sí / no).
   - Antigüedad del cliente, en días.
   - **LTV** (valor económico del cliente): lo que ha pagado durante su relación con Interconnect.
3. **Análisis por segmento.** Se revisaron las cancelaciones por tipo de contrato y por año de alta, para localizar en qué parte de la cartera se concentra el problema.
4. **Validación estadística.** Se comprobó que las diferencias observadas entre clientes activos y clientes cancelados no son producto del azar.

**El modelo** que se construirá en la siguiente fase funcionará como la lista de prioridades de una sala de urgencias. No atiende a todos los pacientes por igual. atiende primero a los de mayor riesgo. Aquí, los "pacientes" son clientes y la "urgencia" es la probabilidad de cancelación.

**Sobre la métrica.** Para evaluar el modelo se usa una métrica estándar de la industria, **AUC-ROC**. Es un número entre 0 y 1 que mide qué tan bien el modelo ordena a los clientes por riesgo: **1.0 sería un ordenamiento perfecto y 0.5 equivale a ordenar al azar**.

## 5. Resultados medibles

### 5.1 Impacto económico

La cancelación de clientes ha costado a Interconnect alrededor de **2.7 millones de dólares** del valor acumulado de su cartera.

| Tipo de contrato | Pérdida de LTV en el segmento |
|------------------|------------------------------:|
| Month-to-month   | 30 %                          |
| One year         | 13 %                          |
| Two year         | 4 %                           |
| **Total cartera**| **15 %**                      |

Del total perdido, **alrededor de 1.8 millones** vienen del segmento mensual. Dos terceras partes de la fuga se concentran ahí.



### 5.2 Evolución de la cancelación en contratos mensuales

| Año de alta | Tasa de cancelación |
|-------------|--------------------:|
| 2014        | 28 %                |
| 2019        | 55 %                |

Las cancelaciones en los contratos de uno y dos años se mantuvieron estables y bajas.

### 5.3 Antigüedad

Los clientes activos llevan, en mediana, aproximadamente **el doble de tiempo** con Interconnect que los clientes que cancelaron.

### 5.4 Validación estadística

La diferencia de antigüedad entre activos y cancelados **no es producto del azar**: la prueba estadística la confirma con un p-value prácticamente en cero, y el tamaño del efecto (~0.69) es grande. Con esto queda validada la base para apoyar la estrategia de retención en la antigüedad del cliente.

### 5.5 Desempeño del modelo

Se evaluaron 8 modelos (Dummy como baseline + 7 algoritmos), con tuning real para XGBoost y un ensemble por promedio de los 3 mejores por validación cruzada. Cada modelo se valida con dos métricas:

- **Sin CV (test):** el modelo se entrena una vez y se evalúa en el conjunto de prueba reservado (1 761 clientes).
- **Con CV (validación cruzada):** el modelo se entrena cinco veces con particiones distintas y se promedia el AUC.

Una diferencia pequeña entre ambos números confirma que no hay sobreajuste.

**Tabla comparativa (AUC-ROC en test):**

| Modelo                                             | AUC test | AUC CV (5-fold) |
|----------------------------------------------------|---------:|----------------:|
| XGBoost tuneado                                    | 0.848    | 0.857           |
| **Ensemble top 3 (XGBoost + CatBoost + RF)**       | **0.847**| **0.859 +/- 0.007** |
| LogisticRegression                                 | 0.846    | 0.849 +/- 0.010 |
| RandomForest                                       | 0.845    | 0.852 +/- 0.007 |
| LightGBM                                           | 0.842    | 0.850 +/- 0.006 |
| CatBoost                                           | 0.836    | 0.855 +/- 0.008 |
| XGBoost base                                       | 0.828    | 0.842 +/- 0.008 |
| DecisionTree                                       | 0.805    | 0.827 +/- 0.011 |
| MLP (sklearn)                                      | 0.798    | 0.840 +/- 0.009 |
| DummyClassifier (baseline)                         | 0.500    | —               |

**Lectura.** El ganador por métrica es el **ensemble de top 3** (XGBoost tuneado + CatBoost + Random Forest): ordena correctamente 85.9 % de los pares de clientes por riesgo y supera con margen el umbral mínimo del sprint (0.75). El mejor modelo individual es **XGBoost tuneado** (0.848 en prueba). La diferencia entre CV y test es ≤ 0.04 en todos los modelos; los números son estables.

### 5.6 Variables principales del riesgo

Las variables con mayor peso en la predicción, por consenso de los seis modelos:

1. **Tipo de contrato** (un contrato a dos años reduce drásticamente el riesgo).
2. **Antigüedad** del cliente.
3. **Cargos mensuales** (tarifas altas asociadas a mayor fuga).
4. **Tipo de servicio de internet** (Fiber optic correlaciona con más churn).
5. **Servicios de valor agregado** (seguridad, soporte, respaldo) como frenos.

### 5.7 Lista de retención para marketing

Se generó un archivo con los **5 174 clientes activos** segmentados por riesgo:

| Segmento | Clientes | % | Acción recomendada                                    |
|----------|---------:|--:|-------------------------------------------------------|
| **Alto** |      776 |15 %| Contacto inmediato + oferta de retención             |
| Medio    |    1 294 |25 %| Email personalizado con incentivo de permanencia     |
| Bajo     |    3 104 |60 %| Monitoreo pasivo, campaña genérica trimestral        |

El archivo se entrega en CSV y Excel (`lista_retencion.csv` / `lista_retencion.xlsx`) listo para que marketing lo opere sin pasar por tecnología.

> **Nota.** La lista actual se genera con LightGBM tuneado (AUC 0.838 test / 0.850 CV) por su rapidez de inferencia.

## 6. Limitaciones y supuestos

- La información es una **fotografía al 1 de febrero de 2020**. No hay datos posteriores para confirmar tendencias.
- El **LTV es una estimación**: cargo mensual por meses activos, sin descuentos ni promociones.
- Los clientes dados de alta en 2020 aun no habian tenido tiempo suficiente para cancelar a la fecha de corte. El 0 % en esa fecha refleja observacion corta, no la realidad.
- Cuando un cliente no aparece en los archivos de internet o telefono, se asume que no contrato ese servicio.
- El **desbalance entre clases** (73 % / 27 %) se compensó con upsampling solo en entrenamiento; la validación se hace sobre datos originales.
- **Techo de precisión (~0.85-0.86).** El objetivo ambicioso del sprint (AUC 0.88) no se alcanza con los datos disponibles. Agotamos las palancas del dataset actual: tuning, ensemble, feature engineering. El margen de 3 puntos porcentuales requiere fuentes nuevas: consumo mensual de datos, historial de quejas, interacciones con soporte técnico. Con acceso a esas fuentes el modelo podria subir a 0.88-0.90 en la siguiente iteración.

## 7. Siguientes pasos y recomendaciones concretas

### 7.1 Recomendación inmediata, independiente del modelo

**Migrar clientes del contrato mensual a contratos anuales o bianuales.** Ahí se concentran dos terceras partes de la fuga y las modalidades largas muestran tasas de cancelación mucho menores. Descuentos por compromiso y servicios adicionales incluidos por firmar un año recuperan ingreso sin necesidad del modelo.

### 7.2 Operar la lista de retención

- **Ya esta lista:** archivo `lista_retencion.xlsx` con 5 174 clientes activos ordenados por probabilidad de cancelación.
- **Segmento Alto** (776 clientes, top 15 %): campaña de contacto telefonico con oferta de retención + upgrade.
- **Segmento Medio** (1 294 clientes, siguiente 25 %): email personalizado con incentivo de permanencia.
- **Segmento Bajo** (3 104 clientes, 60 % restante): monitoreo pasivo, campaña generica trimestral.

### 7.3 Decisiones que requiere la mesa directiva

1. **Presupuesto de retención.** El costo de la campaña Alto define cuánto LTV se recupera. Referencia: atacar el top 776 y rescatar solo el 20 % equivale a ~540 mil USD en LTV preservado.
2. **Umbral de contacto.** Por defecto se usó el percentil 85 (top 15 % como Alto). Si el equipo de retención tiene más o menos capacidad, ese umbral se ajusta.
3. **Frecuencia de actualizacion.** Se recomienda recalcular la lista **mensualmente** usando los archivos actualizados.
4. **Siguiente versión del modelo.** Incorporar variables de comportamiento (uso mensual de datos, historial de quejas, interacciones con soporte) subiria el AUC de 0.85 hacia 0.88-0.90. Requiere integrar fuentes adicionales no disponibles en esta fase.

### 7.4 Transparencia técnica

Durante el desarrollo se detectó y corrigió una **fuga de información en la variable de antigüedad** que, de haberse dejado, habría inflado artificialmente el AUC hasta 1.0. El número real es **0.85** (documentado en la retrospectiva del cuaderno principal). Esta transparencia es clave porque el modelo se usara para decisiones comerciales: los números que se ven aqui son los números reales que dara en produccion.

### Retrospectiva: retos y cómo se resolvieron
- **Graficas.**  Al crear las graficas y no tener datos en algunos meses, se perdio tiempo investigando la causa.
- **Desbalance 73 / 27.** Sobremuestreo solo en entrenamiento; el test conserva la proporción real.
- **Valores en cero.** Al sacar las estadisticas se exploraron los datos para entender esos 11 valores nulos los pasos a seguir.
- **Fuga por `DaysActive`.** La variable codificaba la cancelación y llevaba a AUC 1.0. Se sustituyó por `Tenure`; AUC  0.84.
- **Derivadas del target.** `TotalCharges` y `LTV` usan la fecha de cancelación; se excluyeron del modelado.
- **CV limpia.** Se hizo sobre el train crudo con pesos de clase; el upsampling quedó fuera del CV.
- **Techo en 0.86.** Con variables de negocio y ensemble top-3 se llegó a 0.851 en test.